# Dataset Cleaning & Preparation for Comparability
## Creating Pure Sentiment Classification Datasets

**Project**: DCIT316 - Twi Cross-Dataset Generalization  
**Date**: 2026-09-01  
**Objective**: Remove non-sentiment labels from AfriSenti-Twi and create comparable processed datasets

---

## Problem Statement

The original AfriSenti-Twi dataset contains **8 label types**, but only **3 are pure sentiment labels**:

### Non-Sentiment Labels (to be REMOVED):
- **Ghanaian Pidgin** - Language identification, not sentiment
- **Multilingual** - Language identification, not sentiment
- **No Sentiment** - Absence of sentiment (not a valid class)
- **Twi** - Language tag, not sentiment
- **Positive & Negative** - Ambiguous mixed labels

### Pure Sentiment Labels (to be KEPT):
- **Positive** - Valid sentiment
- **Negative** - Valid sentiment
- **Neutral** - Valid sentiment

## Approach
1. Load both datasets
2. Analyze and identify non-sentiment labels
3. Filter out non-sentiment rows
4. Create comparable processed datasets
5. Document exclusions

## Setup & Dependencies

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set paths
raw_dir = Path('../03_Data/raw')
processed_dir = Path('../03_Data/processed')

# Create directories
processed_dir.mkdir(exist_ok=True, parents=True)

print("✓ Setup complete")
print(f"✓ Raw data directory: {raw_dir}")
print(f"✓ Processed data directory: {processed_dir}")

✓ Setup complete
✓ Raw data directory: ..\03_Data\raw
✓ Processed data directory: ..\03_Data\processed


## Load Raw Datasets

In [2]:
# Load AfriSenti-Twi
afrisenti_df = pd.read_csv(raw_dir / 'afrisenti_twi' / 'labeledl_twi_tweets.csv', encoding='latin-1')

print("=" * 80)
print("LOADING DATASETS")
print("=" * 80)

print(f"\n✓ AfriSenti-Twi loaded: {afrisenti_df.shape}")
print(f"  Columns: {afrisenti_df.columns.tolist()}")
print(f"\nSample data:")
print(afrisenti_df.head())

LOADING DATASETS

✓ AfriSenti-Twi loaded: (5996, 2)
  Columns: ['tweets', 'labels']

Sample data:
                      tweets    labels
0               wode maya ??   Neutral
1       nsuo de ebi poseidon   Neutral
2  ???????????? bra mike nie   Neutral
3             ?y? ?k?m menua  Negative
4                 wo nso saa  Negative


In [3]:
# Load AsanteTwiSenti (all splits)
asantetwi_train = pd.read_csv(raw_dir / 'asantetwi' / 'train.tsv', sep='\t')
asantetwi_dev = pd.read_csv(raw_dir / 'asantetwi' / 'dev.tsv', sep='\t')
asantetwi_test = pd.read_csv(raw_dir / 'asantetwi' / 'test.tsv', sep='\t')
asantetwi_all = pd.concat([asantetwi_train, asantetwi_dev, asantetwi_test], ignore_index=True)

print(f"✓ AsanteTwiSenti loaded:")
print(f"  Train: {asantetwi_train.shape}")
print(f"  Dev: {asantetwi_dev.shape}")
print(f"  Test: {asantetwi_test.shape}")
print(f"  Total: {asantetwi_all.shape}")
print(f"\nSample data (Train):")
print(asantetwi_train.head())

✓ AsanteTwiSenti loaded:
  Train: (3481, 2)
  Dev: (388, 2)
  Test: (949, 2)
  Total: (4818, 2)

Sample data (Train):
                                               tweet     label
0                        kako be shark but wo ti ewu  negative
1            br ne bayie nti na me supporti man city  negative
2  s3 woofis3 mada wafutuo tantan no 3y3wo s3mafa...  negative
3          wabɔdam anaa wo trumu yɛ nkate nkwan aseɛ  negative
4                                  enfa bi da bra 🤣🤣  negative


---

# Step 1: Analyze Label Categories

In [4]:
print("=" * 80)
print("ANALYZING LABEL CATEGORIES IN AfriSenti-Twi")
print("=" * 80)

# Get all unique labels
all_labels = afrisenti_df['labels'].unique()
print(f"\nTotal unique labels: {len(all_labels)}")
print(f"Labels: {sorted(all_labels)}")

# Get label distribution
label_dist = afrisenti_df['labels'].value_counts()
print(f"\nLabel Distribution:")
for label, count in label_dist.items():
    pct = count / len(afrisenti_df) * 100
    print(f"  {label:20s}: {count:5,d} ({pct:5.2f}%)")

ANALYZING LABEL CATEGORIES IN AfriSenti-Twi

Total unique labels: 8
Labels: ['Ghanaian Pidgin', 'Multilingual', 'Negative', 'Neutral', 'No Sentiment', 'Positive', 'Positive & Negative', 'Twi']

Label Distribution:
  Positive            : 3,542 (59.07%)
  Neutral             : 1,308 (21.81%)
  Negative            : 1,064 (17.75%)
  No Sentiment        :    49 ( 0.82%)
  Positive & Negative :    26 ( 0.43%)
  Multilingual        :     3 ( 0.05%)
  Twi                 :     3 ( 0.05%)
  Ghanaian Pidgin     :     1 ( 0.02%)


---

# Step 2: Identify & Categorize Labels

In [5]:
# Define label categories
pure_sentiment_labels = ['Positive', 'Negative', 'Neutral']
non_sentiment_labels = ['Ghanaian Pidgin', 'Multilingual', 'No Sentiment', 'Twi', 'Positive & Negative']

# Define reasons for exclusion
exclusion_reasons = {
    'Ghanaian Pidgin': 'Language identification (Pidgin vs Twi), not sentiment',
    'Multilingual': 'Language identification (mixed languages), not sentiment',
    'No Sentiment': 'Absence of sentiment expression, not a valid sentiment class',
    'Twi': 'Language identification (Twi language tag), not sentiment',
    'Positive & Negative': 'Ambiguous mixed sentiment, unclear binary classification'
}

print("=" * 80)
print("LABEL CATEGORIZATION")
print("=" * 80)

print("\n" + "-" * 80)
print("NON-SENTIMENT LABELS (Will be REMOVED)")
print("-" * 80)
print("These are linguistic metadata or problematic labels:\n")

total_removed = 0
for label in non_sentiment_labels:
    if label in afrisenti_df['labels'].values:
        count = (afrisenti_df['labels'] == label).sum()
        pct = count / len(afrisenti_df) * 100
        print(f"  ❌ {label:20s}: {count:5,d} ({pct:5.2f}%)")
        print(f"     Reason: {exclusion_reasons[label]}\n")
        total_removed += count

print("\n" + "-" * 80)
print("PURE SENTIMENT LABELS (Will be KEPT)")
print("-" * 80)
print("These are valid sentiment expressions:\n")

total_kept = 0
for label in pure_sentiment_labels:
    if label in afrisenti_df['labels'].values:
        count = (afrisenti_df['labels'] == label).sum()
        pct = count / len(afrisenti_df) * 100
        print(f"  ✅ {label:20s}: {count:5,d} ({pct:5.2f}%)")
        total_kept += count

print("\n" + "-" * 80)
print("SUMMARY")
print("-" * 80)
print(f"  Rows to KEEP (pure sentiment):    {total_kept:,d} ({total_kept/len(afrisenti_df)*100:.1f}%)")
print(f"  Rows to REMOVE (non-sentiment):   {total_removed:,d} ({total_removed/len(afrisenti_df)*100:.1f}%)")
print(f"  Total original:                   {len(afrisenti_df):,d}")

LABEL CATEGORIZATION

--------------------------------------------------------------------------------
NON-SENTIMENT LABELS (Will be REMOVED)
--------------------------------------------------------------------------------
These are linguistic metadata or problematic labels:

  ❌ Ghanaian Pidgin     :     1 ( 0.02%)
     Reason: Language identification (Pidgin vs Twi), not sentiment

  ❌ Multilingual        :     3 ( 0.05%)
     Reason: Language identification (mixed languages), not sentiment

  ❌ No Sentiment        :    49 ( 0.82%)
     Reason: Absence of sentiment expression, not a valid sentiment class

  ❌ Twi                 :     3 ( 0.05%)
     Reason: Language identification (Twi language tag), not sentiment

  ❌ Positive & Negative :    26 ( 0.43%)
     Reason: Ambiguous mixed sentiment, unclear binary classification


--------------------------------------------------------------------------------
PURE SENTIMENT LABELS (Will be KEPT)
-----------------------------------------

---

# Step 3: Filter & Clean Datasets

In [6]:
print("\n=" * 80)
print("FILTERING AfriSenti-Twi")
print("=" * 80)

# Filter to only sentiment labels
afrisenti_filtered = afrisenti_df[afrisenti_df['labels'].isin(pure_sentiment_labels)].copy()

print(f"\nOriginal rows:  {len(afrisenti_df):,d}")
print(f"Filtered rows:  {len(afrisenti_filtered):,d}")
print(f"Removed:        {len(afrisenti_df) - len(afrisenti_filtered):,d} ({(len(afrisenti_df) - len(afrisenti_filtered))/len(afrisenti_df)*100:.1f}%)")

print(f"\nFiltered label distribution:")
for label in sorted(afrisenti_filtered['labels'].unique()):
    count = (afrisenti_filtered['labels'] == label).sum()
    pct = count / len(afrisenti_filtered) * 100
    print(f"  {label:20s}: {count:5,d} ({pct:5.1f}%)")


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
FILTERING AfriSenti-Twi

Original rows:  5,996
Filtered rows:  5,914
Removed:        82 (1.4%)

Filtered label distribution:
  Negative            : 1,064 ( 18.0%)
  Neutral             : 1,308 ( 22.1%)
  Positive            : 3,542 ( 59.9%)


In [7]:
# Prepare AfriSenti for processing
print("\nPreparing AfriSenti-Twi for output...")

afrisenti_processed = afrisenti_filtered.copy()
afrisenti_processed = afrisenti_processed[['tweets', 'labels']]
afrisenti_processed.columns = ['text', 'label']
afrisenti_processed['label'] = afrisenti_processed['label'].str.lower()

# Remove any NaN values (if any)
afrisenti_processed = afrisenti_processed.dropna()

print(f"  Shape: {afrisenti_processed.shape}")
print(f"  Columns: {afrisenti_processed.columns.tolist()}")
print(f"\nSample rows:")
print(afrisenti_processed.head())


Preparing AfriSenti-Twi for output...
  Shape: (5914, 2)
  Columns: ['text', 'label']

Sample rows:
                        text     label
0               wode maya ??   neutral
1       nsuo de ebi poseidon   neutral
2  ???????????? bra mike nie   neutral
3             ?y? ?k?m menua  negative
4                 wo nso saa  negative


In [8]:
# Prepare AsanteTwiSenti for processing
print("\nPreparing AsanteTwiSenti for output...")

asantetwi_processed = asantetwi_all.copy()
asantetwi_processed = asantetwi_processed[['tweet', 'label']]
asantetwi_processed.columns = ['text', 'label']

# Remove any NaN values (if any)
asantetwi_processed = asantetwi_processed.dropna()

print(f"  Shape: {asantetwi_processed.shape}")
print(f"  Columns: {asantetwi_processed.columns.tolist()}")
print(f"\nSample rows:")
print(asantetwi_processed.head())


Preparing AsanteTwiSenti for output...
  Shape: (4818, 2)
  Columns: ['text', 'label']

Sample rows:
                                                text     label
0                        kako be shark but wo ti ewu  negative
1            br ne bayie nti na me supporti man city  negative
2  s3 woofis3 mada wafutuo tantan no 3y3wo s3mafa...  negative
3          wabɔdam anaa wo trumu yɛ nkate nkwan aseɛ  negative
4                                  enfa bi da bra 🤣🤣  negative


---

# Step 4: Verify & Compare

In [9]:
print("=" * 80)
print("CLEANED DATASET STATISTICS")
print("=" * 80)

print(f"\nAfriSenti-Twi (CLEANED - Sentiment Labels Only):")
print(f"  Total rows: {len(afrisenti_processed):,d}")
for label in sorted(afrisenti_processed['label'].unique()):
    count = (afrisenti_processed['label'] == label).sum()
    pct = count / len(afrisenti_processed) * 100
    print(f"    • {label:10s}: {count:4,d} ({pct:5.1f}%)")

print(f"\nAsanteTwiSenti (No filtering needed - Pure sentiment labels):")
print(f"  Total rows: {len(asantetwi_processed):,d}")
for label in sorted(asantetwi_processed['label'].unique()):
    count = (asantetwi_processed['label'] == label).sum()
    pct = count / len(asantetwi_processed) * 100
    print(f"    • {label:10s}: {count:4,d} ({pct:5.1f}%)")

CLEANED DATASET STATISTICS

AfriSenti-Twi (CLEANED - Sentiment Labels Only):
  Total rows: 5,914
    • negative  : 1,064 ( 18.0%)
    • neutral   : 1,308 ( 22.1%)
    • positive  : 3,542 ( 59.9%)

AsanteTwiSenti (No filtering needed - Pure sentiment labels):
  Total rows: 4,818
    • negative  : 1,815 ( 37.7%)
    • neutral   :  726 ( 15.1%)
    • positive  : 2,277 ( 47.3%)


In [10]:
# Create comparison table
print("\n" + "-" * 80)
print("FINAL DATASET COMPARISON")
print("-" * 80)

comparison = pd.DataFrame({
    'Metric': ['Total Examples', 'Positive', 'Negative', 'Neutral', 'Rows Removed', '% Cleaned'],
    'AfriSenti-Twi': [
        len(afrisenti_processed),
        f"{(afrisenti_processed['label'] == 'positive').sum():,d}",
        f"{(afrisenti_processed['label'] == 'negative').sum():,d}",
        f"{(afrisenti_processed['label'] == 'neutral').sum():,d}",
        f"{len(afrisenti_df) - len(afrisenti_filtered):,d}",
        f"{(len(afrisenti_df) - len(afrisenti_filtered))/len(afrisenti_df)*100:.1f}%"
    ],
    'AsanteTwiSenti': [
        len(asantetwi_processed),
        f"{(asantetwi_processed['label'] == 'positive').sum():,d}",
        f"{(asantetwi_processed['label'] == 'negative').sum():,d}",
        f"{(asantetwi_processed['label'] == 'neutral').sum():,d}",
        '0',
        '0.0%'
    ]
})

print(comparison.to_string(index=False))


--------------------------------------------------------------------------------
FINAL DATASET COMPARISON
--------------------------------------------------------------------------------
        Metric AfriSenti-Twi AsanteTwiSenti
Total Examples          5914           4818
      Positive         3,542          2,277
      Negative         1,064          1,815
       Neutral         1,308            726
  Rows Removed            82              0
     % Cleaned          1.4%           0.0%


---

# Step 5: Save Processed Datasets

In [11]:
print("\n=" * 80)
print("SAVING PROCESSED DATASETS")
print("=" * 80)

# Save AfriSenti
afrisenti_output = processed_dir / 'afrisenti_twi.csv'
afrisenti_processed.to_csv(afrisenti_output, index=False)
print(f"\n✓ AfriSenti-Twi saved:")
print(f"  File: {afrisenti_output}")
print(f"  Rows: {len(afrisenti_processed):,d}")
print(f"  Size: {afrisenti_output.stat().st_size / 1024:.1f} KB")

# Save AsanteTwiSenti
asantetwi_output = processed_dir / 'asantetwi_senti.csv'
asantetwi_processed.to_csv(asantetwi_output, index=False)
print(f"\n✓ AsanteTwiSenti saved:")
print(f"  File: {asantetwi_output}")
print(f"  Rows: {len(asantetwi_processed):,d}")
print(f"  Size: {asantetwi_output.stat().st_size / 1024:.1f} KB")


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
SAVING PROCESSED DATASETS

✓ AfriSenti-Twi saved:
  File: ..\03_Data\processed\afrisenti_twi.csv
  Rows: 5,914
  Size: 282.2 KB

✓ AsanteTwiSenti saved:
  File: ..\03_Data\processed\asantetwi_senti.csv
  Rows: 4,818
  Size: 284.0 KB


---

# Step 6: Documentation

In [12]:
# Create detailed documentation
doc_path = processed_dir / 'DATA_PROCESSING_LOG.md'

doc_content = f"""# Data Processing Log

## Objective
Prepare comparable sentiment classification datasets by ensuring both AfriSenti-Twi and AsanteTwiSenti use the same target labels: **positive, negative, neutral**.

## Findings

### AfriSenti-Twi: Non-Sentiment Labels Removed

The original AfriSenti-Twi dataset contained **8 distinct label categories**, but only **3 are pure sentiment labels**:

| Label | Count | % | Status | Reason |
|-------|-------|---|--------|--------|
| Ghanaian Pidgin | 1 | 0.0% | ❌ REMOVED | Language identification (Pidgin vs Twi), not sentiment |
| Multilingual | 3 | 0.1% | ❌ REMOVED | Language identification (mixed languages), not sentiment |
| No Sentiment | 49 | 0.8% | ❌ REMOVED | Absence of sentiment expression, not a valid sentiment class |
| Twi | 3 | 0.1% | ❌ REMOVED | Language identification (Twi language tag), not sentiment |
| Positive & Negative | 26 | 0.4% | ❌ REMOVED | Ambiguous mixed sentiment, unclear binary classification |
| **Negative** | 1,064 | 17.7% | ✅ KEPT | Valid sentiment label |
| **Neutral** | 1,308 | 21.8% | ✅ KEPT | Valid sentiment label |
| **Positive** | 3,542 | 59.1% | ✅ KEPT | Valid sentiment label |

### Exclusion Statistics

- **Original AfriSenti-Twi**: {len(afrisenti_df):,d} examples
- **Non-sentiment rows removed**: {len(afrisenti_df) - len(afrisenti_filtered):,d} examples ({(len(afrisenti_df) - len(afrisenti_filtered))/len(afrisenti_df)*100:.1f}%)
- **Cleaned AfriSenti-Twi**: {len(afrisenti_processed):,d} examples ({len(afrisenti_processed)/len(afrisenti_df)*100:.1f}%)

### AsanteTwiSenti: No Filtering Required

AsanteTwiSenti already uses only 3 pure sentiment labels:
- positive: {(asantetwi_processed['label'] == 'positive').sum():,d} ({(asantetwi_processed['label'] == 'positive').sum()/len(asantetwi_processed)*100:.1f}%)
- negative: {(asantetwi_processed['label'] == 'negative').sum():,d} ({(asantetwi_processed['label'] == 'negative').sum()/len(asantetwi_processed)*100:.1f}%)
- neutral: {(asantetwi_processed['label'] == 'neutral').sum():,d} ({(asantetwi_processed['label'] == 'neutral').sum()/len(asantetwi_processed)*100:.1f}%)
- **Total**: {len(asantetwi_processed):,d} examples

## Final Datasets

Both processed datasets have identical structure:

```
text,label
"example tweet text",positive
"another example",negative
...
```

### Dataset Comparison

| Metric | AfriSenti-Twi | AsanteTwiSenti |
|--------|------------------|----------------|
| Total examples | {len(afrisenti_processed):,d} | {len(asantetwi_processed):,d} |
| Positive | {(afrisenti_processed['label'] == 'positive').sum():,d} ({(afrisenti_processed['label'] == 'positive').sum()/len(afrisenti_processed)*100:.1f}%) | {(asantetwi_processed['label'] == 'positive').sum():,d} ({(asantetwi_processed['label'] == 'positive').sum()/len(asantetwi_processed)*100:.1f}%) |
| Negative | {(afrisenti_processed['label'] == 'negative').sum():,d} ({(afrisenti_processed['label'] == 'negative').sum()/len(afrisenti_processed)*100:.1f}%) | {(asantetwi_processed['label'] == 'negative').sum():,d} ({(asantetwi_processed['label'] == 'negative').sum()/len(asantetwi_processed)*100:.1f}%) |
| Neutral | {(afrisenti_processed['label'] == 'neutral').sum():,d} ({(afrisenti_processed['label'] == 'neutral').sum()/len(afrisenti_processed)*100:.1f}%) | {(asantetwi_processed['label'] == 'neutral').sum():,d} ({(asantetwi_processed['label'] == 'neutral').sum()/len(asantetwi_processed)*100:.1f}%) |
| Pre-defined splits | None | Train/Val/Test |
| Non-sentiment rows removed | {len(afrisenti_df) - len(afrisenti_filtered):,d} | 0 |

## Conclusion

Both datasets are now suitable for cross-dataset generalization experiments with:
- ✅ Target labels: positive, negative, neutral
- ✅ Data structure: text + label CSV format
- ✅ Linguistic variety: Twi language tweets
- ✅ Collection source: X (formerly Twitter)
- ✅ No non-sentiment contamination

The exclusion of non-sentiment categories from AfriSenti-Twi ensures that classification models are trained on valid sentiment expressions only, improving label consistency and model interpretability.
"""

with open(doc_path, 'w', encoding='utf-8') as f:
    f.write(doc_content)

print(f"\n✓ Documentation saved:")
print(f"  File: {doc_path}")


✓ Documentation saved:
  File: ..\03_Data\processed\DATA_PROCESSING_LOG.md


---

# Final Verification

In [13]:
print("\n=" * 80)
print("VERIFICATION - LOADING PROCESSED FILES")
print("=" * 80)

# Verify AfriSenti
afri_verify = pd.read_csv(afrisenti_output)
print(f"\nAfriSenti-Twi (processed):")
print(f"  Shape: {afri_verify.shape}")
print(f"  Columns: {afri_verify.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(afri_verify.head())
print(f"\nLabel value counts:")
print(afri_verify['label'].value_counts())


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
VERIFICATION - LOADING PROCESSED FILES

AfriSenti-Twi (processed):
  Shape: (5914, 2)
  Columns: ['text', 'label']

First 5 rows:
                        text     label
0               wode maya ??   neutral
1       nsuo de ebi poseidon   neutral
2  ???????????? bra mike nie   neutral
3             ?y? ?k?m menua  negative
4                 wo nso saa  negative

Label value counts:
label
positive    3542
neutral     1308
negative    1064
Name: count, dtype: int64


In [14]:
# Verify AsanteTwiSenti
as_verify = pd.read_csv(asantetwi_output)
print(f"\nAsanteTwiSenti (processed):")
print(f"  Shape: {as_verify.shape}")
print(f"  Columns: {as_verify.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(as_verify.head())
print(f"\nLabel value counts:")
print(as_verify['label'].value_counts())


AsanteTwiSenti (processed):
  Shape: (4818, 2)
  Columns: ['text', 'label']

First 5 rows:
                                                text     label
0                        kako be shark but wo ti ewu  negative
1            br ne bayie nti na me supporti man city  negative
2  s3 woofis3 mada wafutuo tantan no 3y3wo s3mafa...  negative
3          wabɔdam anaa wo trumu yɛ nkate nkwan aseɛ  negative
4                                  enfa bi da bra 🤣🤣  negative

Label value counts:
label
positive    2277
negative    1815
neutral      726
Name: count, dtype: int64


In [ ]:
# Quality checks
print("\n" + "=" * 80)
print("QUALITY ASSURANCE CHECKS")
print("=" * 80)

print("\nAfriSenti-Twi Checks:")
print(f"  ✓ No missing values: {afri_verify.isnull().sum().sum() == 0}")
print(f"  ✓ Only 3 labels: {set(afri_verify['label'].unique()) == {'positive', 'negative', 'neutral'}}")
print(f"  ✓ No empty texts: {(afri_verify['text'].str.len() > 0).all()}")

print("\nAsanteTwiSenti Checks:")
print(f"  ✓ No missing values: {as_verify.isnull().sum().sum() == 0}")
print(f"  ✓ Only 3 labels: {set(as_verify['label'].unique()) == {'positive', 'negative', 'neutral'}}")
print(f"  ✓ No empty texts: {(as_verify['text'].str.len() > 0).all()}")

print("\n" + "=" * 80)
print("✓ DATA PROCESSING COMPLETE!")
print("=" * 80)
print(f"\nGenerated files:")
print(f"  1. {afrisenti_output}")
print(f"  2. {asantetwi_output}")
print(f"  3. {doc_path}")
print(f"\nBoth datasets are ready for cross-dataset generalization experiments!")


QUALITY ASSURANCE CHECKS

AfriSenti-Twi Checks:
  ✓ No missing values: True
  ✓ Only 3 labels: True
  ✓ No empty texts: True

AsanteTwiSenti Checks:
  ✓ No missing values: True
  ✓ Only 3 labels: True
  ✓ No empty texts: True

✓ DATA PROCESSING COMPLETE!

Generated files:
  1. ..\03_Data\processed\afrisenti_twi.csv
  2. ..\03_Data\processed\asantetwi_senti.csv
  3. ..\03_Data\processed\DATA_PROCESSING_LOG.md

Both datasets are ready for cross-dataset generalization experiments! 🚀
